# RQ1 — Wikidata result-equivalent query retrieval

**Primary evaluation:** full test set (`wikidata_testset_RQ1.json`, 1063 anchors), relevance = result-equivalent positives (answer-Jaccard > 0.8, binary), all text baselines read the same label-expanded queries SparqlCE reads. Lexical similarity is used **only** to stratify anchors (the "where does each method win" analysis), never as relevance.

**Supplementary (Section 4):** result-wise *graded* relevance from actual answer sets, on the subset with available result sets — a robustness check, not the headline (result sets cover ~57% of anchors).

Run top to bottom.

In [ ]:
# ── 0. Setup ────────────────────────────────────────────────────────────────
import os, sys, json
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(_PROJECT_ROOT))
_RQ1_DIR = Path.cwd().parents[0]  # notebooks/evaluation-RQ1 (holds evaluate_rq1.py)
if str(_RQ1_DIR) not in sys.path: sys.path.insert(0, str(_RQ1_DIR))

import evaluate_rq1
from evaluate_rq1 import *
from sparqlce.sparql_utils import expand_wikidata_labels as WDX

# Fair comparison: every TEXT baseline reads the SAME label-expanded query text
# SparqlCE reads (P31 -> "instance of (P31)"). SparqlCE expands internally, so it
# takes NO transform.
E5_XFORM  = lambda q: f"query: {WDX(q)}"   # E5 family: "query: " prefix + expand
TXT_XFORM = WDX                             # other encoders / lexical: expand only

TEST_DATA_PATH   = os.path.join(_PROJECT_ROOT, "datasets", "wikidata_testset_RQ1.json")
CHECKPOINT_PATH  = os.path.join(_PROJECT_ROOT, "checkpoints_b128", "wikidata", "seed42_full", "best_model.pt")
E5_FT_PATH       = os.path.join(_PROJECT_ROOT, "checkpoints", "e5_square_wikidata", "best_model")
RESULT_SETS_PATH = os.path.join(_PROJECT_ROOT, "datasets", "result_sets_wikidata.json")

TOPKS = [3, 5, 10, 20]
BATCH = 32
print(f"✓ project root: {_PROJECT_ROOT}")

## 1. Load test set + build the PRIMARY (annotation-based) evaluation

In [ ]:
# ── 1. Primary evaluation objects ───────────────────────────────────────────
# Relevance = annotated positives, which are all result-equivalent (Jaccard>0.8).
# Full coverage (all 1063 anchors), binary. This is the primary benchmark.
test_data = load_triplets(TEST_DATA_PATH)
global_corpus, seen_q      = build_global_corpus(test_data)
eval_items                 = build_eval_items(test_data, global_corpus, seen_q)
relevance_scores_per_item  = build_binary_relevance_maps(eval_items, global_corpus)

# Lexical stratification is a DIAGNOSTIC axis only (cumulative cos < threshold):
#   [0,0.3) adversarial | [0.3,0.5) divergent | [0.5,0.8) moderate | rest near-identical
evaluate_rq1.STRATIFY_BINS = [0.3, 0.5, 0.8]

print(f"Primary eval ready -> anchors: {len(eval_items)} | corpus: {len(global_corpus)}")

## 2. Primary evaluation — all models

Each block prints aggregate MRR/Hits/nDCG **and** the lexical-similarity stratified MRR.

**How to read the winner:** on the *aggregate* the fine-tuned text encoder (E5-FT) leads; in the *adversarial `<0.3` stratified slice* — result-equivalent queries that share almost no surface form, the regime this method targets — **SparqlCE** leads. Read both.

In [ ]:
# SParqlCE (ours) — no transform; expands P/Q ids internally via its graph builder.
evaluate_sparqlce_checkpoint(CHECKPOINT_PATH, eval_items, global_corpus,
                             relevance_scores_per_item, TOPKS, batch_size=BATCH)

In [ ]:
# E5-base-v2 (fine-tuned) — the strong, capacity-matched trained baseline.
evaluate_sentence_transformer(E5_FT_PATH, "E5-base-v2 (fine-tuned)", eval_items,
                              global_corpus, relevance_scores_per_item, TOPKS,
                              transform=E5_XFORM)

In [ ]:
# Lexical + zero-shot text baselines (all label-expanded).
evaluate_tfidf(eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=TXT_XFORM)
evaluate_bm25 (eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=TXT_XFORM)
evaluate_sentence_transformer('intfloat/e5-base-v2', 'E5-base-v2', eval_items,
                              global_corpus, relevance_scores_per_item, TOPKS, transform=E5_XFORM)
evaluate_code_llm('Qwen/Qwen2.5-Coder-0.5B', 'Qwen2.5-Coder-0.5B', eval_items,
                  global_corpus, relevance_scores_per_item, TOPKS, transform=TXT_XFORM)

In [ ]:
# Prior-literature SPARQL/code encoders (zero-shot, mean-pooled): CodeBERT,
# GraphCodeBERT, SPBERT. The "mean pooling" notice each prints is expected.
evaluate_sparql_encoder_baselines(eval_items, global_corpus, relevance_scores_per_item,
                                  TOPKS, batch_size=BATCH, transform=TXT_XFORM)

In [ ]:
# Sentence-transformer baselines.
evaluate_sentence_transformer('sentence-transformers/all-MiniLM-L6-v2', 'SBERT/MiniLM',
                              eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=TXT_XFORM)
evaluate_sentence_transformer('sentence-transformers/all-mpnet-base-v2', 'SBERT/MPNet',
                              eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=TXT_XFORM)

## 3. Significance — SparqlCE vs fine-tuned E5 (paired Wilcoxon)

Per-anchor reciprocal rank on the primary eval; one-sided Wilcoxon that SparqlCE > E5-FT.
To test the **adversarial** claim specifically, filter `eval_items` to the `<0.3` slice first (see comment).

In [ ]:
import numpy as np
from scipy.stats import wilcoxon
from sentence_transformers import SentenceTransformer

def per_anchor_rr(sorted_pools, items):
    rr = []
    for pool, (_a, _p, pos) in zip(sorted_pools, items):
        r = 0.0
        for i, q in enumerate(pool, 1):
            if q in pos: r = 1.0 / i; break
        rr.append(r)
    return np.array(rr)

# (optional) restrict to the adversarial slice to test THAT claim:
# lex = evaluate_rq1._lexical_cos_fn(eval_items)
# idx = [i for i,(a,_,pos) in enumerate(eval_items) if any(lex(a,p) < 0.3 for p in pos)]
# items_sig = [eval_items[i] for i in idx]
items_sig = eval_items

sc = SPARQLEmbedder(CHECKPOINT_PATH)
rr_sc = per_anchor_rr(build_sorted_pools_global(
    build_emb_map_sparqlce(sc, global_corpus), items_sig, global_corpus), items_sig)

e5 = SentenceTransformer(E5_FT_PATH)
rr_e5 = per_anchor_rr(build_sorted_pools_global(
    build_emb_map_sentence_transformer(e5, global_corpus, transform=E5_XFORM),
    items_sig, global_corpus), items_sig)

print(f"mean RR  SparqlCE={rr_sc.mean():.4f}  E5-FT={rr_e5.mean():.4f}  n={len(rr_sc)}")
stat, p = wilcoxon(rr_sc, rr_e5, alternative="greater")
print(f"Wilcoxon (SparqlCE > E5-FT): p = {p:.3e}")

## 4. Supplementary robustness — result-wise (answer-set) graded relevance

Recomputes relevance directly from answer-set overlap (Jaccard), graded, over the whole corpus. **Separate variables** (`rw_*`) — this does NOT touch the primary objects above. Result sets cover ~57% of anchors, so this is a robustness check on the covered subset, reported with `stats`, **not** the headline. Check `stats` (coverage, anchors kept, extra result-equivalents recovered) before trusting the numbers.

In [ ]:
from result_relevance import build_result_wise

rw_eval_items, rw_relevance, stats = build_result_wise(
    eval_items, global_corpus, RESULT_SETS_PATH, tau=0.8)
print(stats)

# Re-run the two key models under result-wise relevance (covered subset).
evaluate_sparqlce_checkpoint(CHECKPOINT_PATH, rw_eval_items, global_corpus,
                             rw_relevance, TOPKS, batch_size=BATCH)
evaluate_sentence_transformer(E5_FT_PATH, "E5-base-v2 (fine-tuned)", rw_eval_items,
                              global_corpus, rw_relevance, TOPKS, transform=E5_XFORM)

## Appendix — result-set coverage diagnostic (optional)

In [ ]:
import re, subprocess
_have = set(json.load(open(os.path.join(_PROJECT_ROOT, "datasets", "result_sets_wikidata_rq1subset.json"))))
_norm = lambda s: " ".join(s.split())
_unm = [e for e in test_data if _norm(e["anchor_query"]) not in _have]
print(f"{len(_unm)}/{len(test_data)} anchors unmatched (no result set matched at build time)")
for e in _unm[:5]:
    q = e["anchor_query"]
    lines = [l.strip() for l in q.split("\n") if re.search(r"[PQ]\d", l)]
    cand = max(lines, key=len) if lines else q[:60]
    hits = subprocess.run(["grep","-c","-F",cand,RESULT_SETS_PATH],
                          capture_output=True, text=True).stdout.strip()
    print(f"hits={hits}  line={cand!r}")